In [1]:
!pip install langchain langchain-neo4j
!pip install langchain langchain-community
!pip install neo4j>=5.21
!pip install python-dotenv>=1.0

In [ ]:
# kg_import_outputjson.py
import os
import json
from typing import List, Dict, Any

from langchain_neo4j import Neo4jGraph
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
from langchain_core.documents import Document

# ===================== CẤU HÌNH NEO4J =====================
NEO4J_URI      = "bolt://localhost:7687"   # ⚠ dùng bolt:// nếu chạy local
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "12345678"                # đổi đúng pass
NEO4J_DATABASE = "neo4j"

# ===================== FILE JSON NGUỒN ====================
JSON_PATH = "C:/Users/seina/Chatbot-KHTC/backend/output_quytrinh.json"

# ===================== LỆNH XÓA TOÀN BỘ KG =====================
WIPE_ALL_CYPHER = "MATCH (n) DETACH DELETE n"

# ===================== BUILD GRAPH DOCUMENTS =====================
def load_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def build_graph_documents(payload: Dict[str, Any]) -> List[GraphDocument]:
    nodes: Dict[str, Node] = {}
    rels: List[Relationship] = []

    # Helper tạo node
    def add_nodes(label: str, items: list, props_map: Dict[str,str]):
        for it in items:
            nid = it["id"]
            props = {pname: it.get(jname) for pname, jname in props_map.items()}
            nodes[nid] = Node(type=label, id=nid, properties=props)

    # Tạo các loại node
    add_nodes("Document", payload.get("Documents", []), {
        "name": "Name", "number_of_pages": "Number_of_Pages", "number_of_chapter": "Number_of_Chapter"
    })
    add_nodes("Chapter", payload.get("Chapters", []), {
        "name": "Name", "number": "Number"
    })
    add_nodes("Section", payload.get("Sections", []), {
        "name": "Name", "number": "Number"
    })
    add_nodes("Subsection", payload.get("Subsections", []), {
        "name": "Name", "letter": "Letter"
    })
    add_nodes("Procedure", payload.get("Procedures", []), {
        "name": "Name", "number": "Number"
    })
    add_nodes("Step", payload.get("Steps", []), {
        "name": "Name", "number": "Number"
    })

    # Tạo quan hệ
    for rel in payload.get("Relationships", []):
        src = nodes.get(rel["from"])
        tgt = nodes.get(rel["to"])
        if src and tgt:
            rels.append(Relationship(source=src, target=tgt, type=rel["type"], properties={}))

    # GraphDocument duy nhất
    src_doc = Document(page_content=f"KG from {os.path.basename(JSON_PATH)}",
                       metadata={"source": JSON_PATH})
    return [GraphDocument(nodes=list(nodes.values()), relationships=rels, source=src_doc)]

# ===================== MAIN =====================
def main():
    print(f"Reading: {JSON_PATH}")
    data = load_json(JSON_PATH)

    graph = Neo4jGraph(
        url=NEO4J_URI,
        username=NEO4J_USER,
        password=NEO4J_PASSWORD,
        database=NEO4J_DATABASE,
        refresh_schema=False,
        enhanced_schema=False,   # không cần APOC
    )

    print("Wiping entire KG...")
    graph.query(WIPE_ALL_CYPHER)

    graph_docs = build_graph_documents(data)
    print(f"Prepared {len(graph_docs)} GraphDocument(s). Importing...")

    graph.add_graph_documents(graph_docs, include_source=False)

    graph.refresh_schema()

    sample = graph.query("""
    MATCH (d:Document)-[:HAS_CHAPTER]->(c:Chapter)
    RETURN d.name AS document, c.number AS chapterNum, c.name AS chapterName
    ORDER BY c.number LIMIT 10
    """)
    print("Sample rows:", sample)
    print("\nDone.")

if __name__ == "__main__":
    main()
